# Generic enums - Rust

All 15 Rust examples from [docs/generic.md](https://platob.github.io/yggdryl/generic/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::generic::Holder;
use yggdryl::io::{Buffer, IOBase};

// A value that could have been any handle. The calls do not change.
let mut handle = Holder::buffer(Buffer::new());
handle.write_all_bytes(b"AAPL,1\n")?;

assert_eq!(handle.read_all_bytes()?, b"AAPL,1\n");
assert_eq!(handle.kind(), yggdryl::IOKind::Memory);

## Holder: every storage handle

In [ ]:
use yggdryl::generic::Holder;

// An existing directory is a folder; anything else is a mapped file.
let directory = Holder::local(std::env::temp_dir())?;
assert!(matches!(directory, Holder::Folder(_)));

let missing = Holder::local(std::env::temp_dir().join("yggdryl-generic-doc.bin"))?;
assert!(matches!(missing, Holder::File(_)));

In [ ]:
use yggdryl::generic::Holder;
use yggdryl::io::IOBase;

let root = Holder::folder(std::env::temp_dir())?;
assert!(root.is_container());

// A child need not exist. Naming one yields a leaf handle, and nothing is created.
let leaf = root.child_by("yggdryl-generic-child.bin")?;
assert!(matches!(leaf, Holder::File(_)));
assert!(!leaf.is_container());
assert_eq!(leaf.size(), 0);

## Codec: a coding over a handle

In [ ]:
use yggdryl::generic::Codec;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::Url;

let named = Buffer::new().with_media_type(Url::from_str("file:///trades.csv.zst")?.media_type());
let mut handle = Codec::infer(named);
assert_eq!(handle.codec(), yggdryl::Codec::Zstd);

handle.write_all_bytes(b"symbol,price\nAAPL,1\nAAPL,2\n")?;
handle.flush()?;

// The coded handle reads plain bytes; the handle underneath holds the frame.
assert_eq!(handle.read_all_bytes()?, b"symbol,price\nAAPL,1\nAAPL,2\n");
assert_ne!(handle.handle().as_slice(), b"symbol,price\nAAPL,1\nAAPL,2\n");

In [ ]:
use yggdryl::generic::Codec;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::Level;

let mut handle = Codec::wrap(Buffer::new(), yggdryl::Codec::Gzip).with_level(Level::BEST);
handle.write_all_bytes(b"symbol,price\nAAPL,1\n")?;

// into_handle publishes the pending write, then gives back the compressed bytes.
let inner = handle.into_handle()?;
assert_eq!(yggdryl::gzip::load(inner.as_slice())?, b"symbol,price\nAAPL,1\n");

In [ ]:
use yggdryl::generic::Codec;
use yggdryl::io::Buffer;

let handle = Codec::wrap(Buffer::new(), yggdryl::Codec::Deflate);
assert_eq!(handle.codec(), yggdryl::Codec::Zlib);

## Media: a record encoding over a handle

In [ ]:
use yggdryl::generic::{Holder, Media};
use yggdryl::io::Buffer;
use yggdryl::Url;

fn named(name: &str) -> Result<Holder, Box<dyn std::error::Error>> {
    let url = Url::from_str(&format!("file:///{name}"))?;
    Ok(Holder::buffer(Buffer::new().with_media_type(url.media_type())))
}

assert!(matches!(Media::open(named("trades.arrows")?)?, Media::Ipc(_)));
assert!(matches!(Media::open(named("trades.parquet")?)?, Media::Parquet(_)));
assert!(matches!(Media::open(named("app.log")?)?, Media::Text(_)));

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::generic::{Holder, Media};
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{DataType, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from(vec![1, 2]))],
)?;

let url = Url::from_str("file:///trades.arrows")?;
let handle = Holder::buffer(Buffer::new().with_media_type(url.media_type()));
let mut media = Media::open(handle)?.with_schema(schema.clone());

media.write_batch_reader(arrow::batch_reader(arrow_schema, [batch]))?;
assert_eq!(media.read_batch_reader(None)?.count(), 1);
assert_eq!(media.schema()?, schema);

// A Media is also the bytes it encodes: an Arrow IPC stream opens with its
// continuation marker.
assert_eq!(media.read_range(0, 4)?, [0xFF, 0xFF, 0xFF, 0xFF]);

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::generic::{Holder, Media};
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{DataType, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from(vec![9]))],
)?;

let url = Url::from_str("file:///trades.arrows.gz")?;
let handle = Holder::buffer(Buffer::new().with_media_type(url.media_type()));
let mut media = Media::open(handle)?.with_schema(schema.clone());

media.write_batch_reader(arrow::batch_reader(arrow_schema, [batch]))?;
assert_eq!(media.read_batch_reader(None)?.count(), 1);

// Still an Arrow IPC stream, now behind gzip framing.
assert_eq!(media.read_range(0, 2)?, [0x1F, 0x8B]);

In [ ]:
use yggdryl::generic::{Holder, Media};
use yggdryl::io::Buffer;
use yggdryl::Url;

let url = Url::from_str("file:///trades.csv")?;
let handle = Holder::buffer(Buffer::new().with_media_type(url.media_type()));

let message = Media::open(handle).unwrap_err().to_string();
assert!(message.contains("text/csv"), "{message}");

## RecordOptions: every encoding's settings

In [ ]:
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::{DataType, MimeType, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

let options = RecordOptions::for_media_type(&Url::from_str("file:///trades.parquet")?.media_type())?
    .with_schema(schema.clone())
    .with_batch_size(1024);

assert_eq!(options.mime_type(), MimeType::PARQUET);
assert_eq!(options.schema(), Some(&schema));
assert_eq!(options.batch_size(), Some(1024));

In [ ]:
use arrow_array::RecordBatch;
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::{DataType, MimeType};

let declared = DataType::from_fields([
    DataType::Utf8.required_field("symbol"),
    DataType::Int64.required_field("price"),
])?
.required_field("row");

let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?
    .with_schema(declared.clone())
    .with_select_by_names(["price"]);

// One call is the whole pipeline: the declared cast, then the selection.
// Passing a stored root as the second argument adds the completion layer.
let batch = RecordBatch::new_empty(yggdryl::arrow::schema_from_field(&declared)?);
let cast = options.cast_arrow_batch(batch, None)?;
assert_eq!(cast.num_columns(), 1);

In [ ]:
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::ipc::IpcOptions;
use yggdryl::MimeType;

let options: RecordOptions = IpcOptions::new().with_root_name("trade").with_safe(false).into();

assert_eq!(options.mime_type(), MimeType::ARROW_STREAM);
assert_eq!(options.root_name(), "trade");
assert!(!options.safe());

In [ ]:
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::MimeType;

let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?;
assert!(options.schema().is_none());

let message = options.require_schema().unwrap_err().to_string();
assert!(message.contains("with_schema"), "{message}");

## TypedValue: one value and its datatype

In [ ]:
use yggdryl::generic::{Int64Value, TypedValue};
use yggdryl::{DataType, Value};

let price = TypedValue::from_parts(DataType::Int64, Value::from(7_i64))?;
assert_eq!(price.data_type(), &DataType::Int64);

// The same pairing, with the datatype fixed at compile time.
let typed: Int64Value = price.try_into_typed()?;
assert_eq!(typed.value(), &Value::I64(7));
assert!(Int64Value::new(Value::from("seven")).is_err());